In [0]:
%sql
create database project.sales

In [0]:
%sql
create volume project.sales.data

In [0]:
raw_orders = spark.read.csv("/Volumes/project/sales/data/bronze/retail_dataset.csv", header=True, inferSchema=True)

display(raw_orders)


order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no
1006,09-16-2025,C028,Customer_28,P110,Watch,Accessories,3,$75.72,upi,cancelled,no
1007,27/07/2025,C064,Customer_64,P116,keyboard,electronics,2,315.59,upi,cancelled,no
1008,2025-08-10,C069,Customer_69,P103,Denim Jean,Apparel,null,$150.28,upi,delivered,no
1009,06/09/2025,C065,Customer_65,P109,Jeans,Apparel,-1,271.99,cash,cancelled,yes
1010,2025-09-18,C031,Customer_31,P109,jeans,Apparel,1,$47.4,card,cancelled,no


In [0]:
raw_customers = spark.read.json("/Volumes/project/sales/data/bronze/customer_dataset.json")
display(raw_customers)

age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06
37.0,Hyderabad,C006,f,null,27-Apr-2018
50.0,Kolkata,C007,M,null,2021/11/02
59.0,Bangalore,C008,null,Silver,17-Jun-2021
22.0,BENGALURU,C009,F,null,06-May-2020
58.0,Hyderabad,C010,M,Gold,12/12/2019


In [0]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.ansi.enabled", "false")

order_ts = coalesce(
    to_timestamp(trim(col("order_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("order_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("order_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("order_date")), "MM-dd-yyyy"),
    to_timestamp(trim(col("order_date")), "dd-MM-yyyy")
)

clean_df = (
    raw_orders
    .withColumn("order_id", trim(col("order_id")).cast("long"))
    .withColumn("order_date_ts", order_ts)
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", lower(trim(col("product_name"))))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("quantity", when(trim(col("quantity")).isNull() | (trim(col("quantity"))==""), lit(1)).otherwise(trim(col("quantity"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("quantity", when(col("quantity") <= 0, lit(1)).otherwise(col("quantity")))
    .withColumn("price_clean", translate(trim(col("price")), "$,", ""))
    .withColumn("price", col("price_clean").cast("double"))
    .drop("price_clean")
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("returned", lower(trim(col("returned"))))
    .withColumn("total_amount", (coalesce(col("quantity"), lit(0)) * coalesce(col("price"), lit(0.0))).cast("double"))
)

clean_df = clean_df.dropDuplicates(["order_id", "product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())

clean_df.write.format("delta").mode("overwrite").saveAsTable("project.sales.silver_orders")


In [0]:
%sql
select * from project.sales.silver_orders

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,order_date_ts,total_amount
1039,08-19-2025,C079,Customer_79,P108,t-shirt l,apparel,1,220.61,cash,cancelled,no,2025-08-19T00:00:00.000Z,220.61
1046,26/07/2025,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,2025-07-26T00:00:00.000Z,943.89
1064,20-Jul-2025,C072,Customer_72,P112,backpack,accessories,3,270.6,cash,delivered,no,2025-07-20T00:00:00.000Z,811.8000000000001
1086,29-Jul-2025,C020,Customer_20,P119,sunglasses,accessories,3,158.56,card,delivered,no,2025-07-29T00:00:00.000Z,475.68
1112,08-12-2025,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,2025-08-12T00:00:00.000Z,558.42
1124,07/08/2025,C024,Customer_24,P108,t-shirt l,apparel,3,91.87,cash,completed,no,2025-08-07T00:00:00.000Z,275.61
1127,10/09/2025,C061,Customer_61,P117,usb cable,accessories,3,382.12,cash,returned,yes,2025-09-10T00:00:00.000Z,1146.3600000000001
1145,09-14-2025,C030,Customer_30,P101,t-shirt xl,apparel,3,139.19,upi,completed,no,2025-09-14T00:00:00.000Z,417.57
1191,2025-07-28,C026,Customer_26,P102,sneakers,shoes,2,453.01,card,shipped,no,2025-07-28T00:00:00.000Z,906.02
1192,31-Jul-2025,C055,Customer_55,P112,backpack,accessories,1,143.53,upi,delivered,no,2025-07-31T00:00:00.000Z,143.53


In [0]:
display(raw_customers.limit(10))

age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06
37.0,Hyderabad,C006,f,null,27-Apr-2018
50.0,Kolkata,C007,M,null,2021/11/02
59.0,Bangalore,C008,null,Silver,17-Jun-2021
22.0,BENGALURU,C009,F,null,06-May-2020
58.0,Hyderabad,C010,M,Gold,12/12/2019


In [0]:
signup_ts = coalesce(
    to_timestamp(trim(col("signup_date")), "dd-MMM-yyyy"),
    to_date(trim(col("signup_date")), "dd/MM/yyyy"),
    to_date(trim(col("signup_date")), "yyyy/MM/dd"),
    to_date(trim(col("signup_date")), "yyyy-MM-dd"),
    to_date(trim(col("signup_date")), "dd-MMM-yyyy")
)

clean_customers = (
    raw_customers
    .withColumn("age", when(trim(col("age")).isNull() | (trim(col("age"))=="") | (trim(col("age"))=="-"),None).otherwise(trim(col("age"))))
    .withColumn("age", when(col("age").cast('int')<0, None).otherwise(col("age").cast('int')))
    .withColumn("city", trim(col("city")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender", lower(trim(col("gender"))))
    .withColumn("loyalty_tier", upper(trim(col("loyalty_tier"))))
    .withColumn("signup_date", signup_ts)
)

clean_customers.write.format("delta").mode("overwrite").saveAsTable("project.sales.silver_customers")



In [0]:
%sql
select* from project.sales.silver_customers

age,city,customer_id,gender,loyalty_tier,signup_date
35,Delhi,C001,m,GOLD,2020-04-01
39,Gurgaon,C002,null,SILVER,2023-11-14
null,Kolkata,C003,f,SILVER,2022-09-13
null,BENGALURU,C004,male,null,2017-02-21
19,Pune,C005,male,PLATINUM,2022-06-06
37,Hyderabad,C006,f,null,2018-04-27
50,Kolkata,C007,m,null,2021-11-02
59,Bangalore,C008,null,SILVER,2021-06-17
22,BENGALURU,C009,f,null,2020-05-06
58,Hyderabad,C010,m,GOLD,2019-12-12


In [0]:
# GOLD: aggregate (product x customer x year x month x date)
silver_orders = spark.table("project.sales.silver_orders")
silver_customers = spark.table("project.sales.silver_customers")
orders_enriched = silver_orders.join(silver_customers, on="customer_id", how="left")
orders_enriched = orders_enriched.withColumn("order_date", to_date(col("order_date_ts"))) \
                                 .withColumn("year", year(col("order_date_ts"))) \
                                 .withColumn("month", month(col("order_date_ts"))) 
                                
gold_df = (
    orders_enriched
    .groupBy("product_id","product_name","customer_id","category","year","month","order_date")
    .agg(
        sum("total_amount").alias("total_sales"),
        sum(coalesce(col("quantity"), lit(0))).alias("total_quantity"),
        countDistinct("order_id").alias("total_orders"),
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        avg("price").alias("avg_unit_price"),
        sum(when(col("returned") == "yes", 1).otherwise(0)).alias("returned_count"),
        sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount"),
       
    )
)

gold_df = gold_df.withColumn("avg_order_value", when(col("total_orders")>0, col("total_sales")/col("total_orders")).otherwise(lit(0.0))) \
                 .withColumn("avg_price_per_item", when(col("total_quantity")>0, col("total_sales")/col("total_quantity")).otherwise(lit(0.0))) \
                 .withColumn("refreshed_at", current_timestamp())

gold_df.write.format("delta").mode("overwrite").saveAsTable("project.sales.gold_aggregates")


In [0]:
%sql
select * from project.sales.gold_aggregates


product_id,product_name,customer_id,category,year,month,order_date,total_sales,total_quantity,total_orders,min_price,max_price,avg_unit_price,returned_count,returned_amount,avg_order_value,avg_price_per_item,refreshed_at
P102,sneakers,C021,shoes,2025,8,2025-08-30,713.32,2,1,356.66,356.66,356.66,1,713.32,713.32,356.66,2026-04-16T01:04:33.849Z
P114,adapter,C050,accessories,2025,9,2025-09-02,373.68,2,1,186.84,186.84,186.84,0,0.0,373.68,186.84,2026-04-16T01:04:33.849Z
P100,t-shirt xl,C068,apparel,2025,8,2025-08-28,808.38,2,1,404.19,404.19,404.19,0,0.0,808.38,404.19,2026-04-16T01:04:33.849Z
P111,socks,C007,apparel,2025,7,2025-07-24,609.12,2,1,304.56,304.56,304.56,0,0.0,609.12,304.56,2026-04-16T01:04:33.849Z
P100,t-shirt xl,C031,apparel,2025,8,2025-08-28,1139.46,3,1,379.82,379.82,379.82,0,0.0,1139.46,379.82,2026-04-16T01:04:33.849Z
P104,headphone,C041,electronics,2025,9,2025-09-01,68.53,1,1,68.53,68.53,68.53,1,68.53,68.53,68.53,2026-04-16T01:04:33.849Z
P114,adapter,C069,accessories,2025,9,2025-09-07,306.99,1,1,306.99,306.99,306.99,0,0.0,306.99,306.99,2026-04-16T01:04:33.849Z
P107,charger,C032,accessories,2025,8,2025-08-01,20.95,1,1,20.95,20.95,20.95,0,0.0,20.95,20.95,2026-04-16T01:04:33.849Z
P113,belt,C069,accessories,2025,8,2025-08-27,740.01,3,1,246.67,246.67,246.67,0,0.0,740.01,246.67,2026-04-16T01:04:33.849Z
P113,belt,C078,accessories,2025,8,2025-08-08,1072.1399999999999,3,1,357.38,357.38,357.38,0,0.0,1072.1399999999999,357.37999999999994,2026-04-16T01:04:33.849Z
